<a href="https://colab.research.google.com/github/Nikk118/Nikk118/blob/main/gpt_from_scratch_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
#self attention

In [25]:
import torch
import torch.nn as nn
import math

class self_attention(nn.Module):
  def __init__(self,d_model):
    super().__init__()
    self.w_q=nn.Linear(d_model,d_model)
    self.w_k=nn.Linear(d_model,d_model)
    self.w_v=nn.Linear(d_model,d_model)

  def forward(self,x):
    print("input...")

    q=self.w_q(x)
    k=self.w_k(x)
    v=self.w_v(x)

    kT=k.transpose(-2,-1)
    score=torch.matmul(q,kT)
    score=score/math.sqrt(x.size(-1))
    attention=torch.softmax(score,dim=-1)
    output=torch.matmul(attention,v)

    return output

x=torch.rand(2,4,8)

sa=self_attention(8)

out=sa(x)
print(out)



input...
tensor([[[-0.8343, -0.5558,  0.5395, -0.6035,  0.4240, -0.4963, -0.4231,
           0.2940],
         [-0.8357, -0.5568,  0.5401, -0.6049,  0.4245, -0.4967, -0.4235,
           0.2936],
         [-0.8351, -0.5558,  0.5395, -0.6039,  0.4236, -0.4967, -0.4210,
           0.2946],
         [-0.8341, -0.5559,  0.5395, -0.6033,  0.4242, -0.4961, -0.4236,
           0.2939]],

        [[-0.7369, -0.3814,  0.5224, -0.2283,  0.3078, -0.3351, -0.1173,
           0.3086],
         [-0.7386, -0.3807,  0.5251, -0.2275,  0.3036, -0.3352, -0.1142,
           0.3071],
         [-0.7362, -0.3820,  0.5215, -0.2286,  0.3095, -0.3351, -0.1189,
           0.3091],
         [-0.7379, -0.3790,  0.5257, -0.2249,  0.3011, -0.3329, -0.1108,
           0.3077]]], grad_fn=<UnsafeViewBackward0>)


#multi head attention


In [26]:
import torch
import torch.nn as nn
import math

class multi_head_attention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()
    if d_model%num_heads!=0:
      raise ValueError("d_model is not in even number")

    self.d_model=d_model
    self.nums_heads=num_heads
    self.head_dim=d_model//num_heads

    self.w_q=nn.Linear(d_model,d_model)
    self.w_k=nn.Linear(d_model,d_model)
    self.w_v=nn.Linear(d_model,d_model)
    self.w_o=nn.Linear(d_model,d_model)

    self.scale=math.sqrt(self.head_dim)

  def forward(self,x):
    batch_size,seq_len,d_model=x.shape
    q=self.w_q(x)
    k=self.w_k(x)
    v=self.w_v(x)

    q=q.view(batch_size,seq_len,self.nums_heads,self.head_dim)
    k=k.view(batch_size,seq_len,self.nums_heads,self.head_dim)
    v=v.view(batch_size,seq_len,self.nums_heads,self.head_dim)

    q=q.transpose(1,2)
    k=k.transpose(1,2)
    v=v.transpose(1,2)
    kT=k.transpose(-2,-1)

    score=torch.matmul(q,kT)/self.scale

    seq_len=score.size(-1)
    mask=torch.tril(torch.ones(seq_len,seq_len)).to(score.device)

    score=score.masked_fill(mask==0,float('-inf'))
    attention=torch.softmax(score,dim=-1)

    output=torch.matmul(attention,v)

    output=output.transpose(1,2)

    output=output.reshape(batch_size,seq_len,d_model)
    output=self.w_o(output)

    return output


In [27]:

x=torch.rand(2,4,8)

m=multi_head_attention(d_model=8,num_heads=2)

result=m(x)
print(result)

tensor([[[ 0.0037, -0.0932,  0.1865, -0.5482, -0.3722,  0.2256, -0.1667,
          -0.1398],
         [ 0.0389, -0.1209,  0.2521, -0.6143, -0.3550,  0.2756, -0.1341,
          -0.1153],
         [-0.0282, -0.1211,  0.2201, -0.5877, -0.3536,  0.2680, -0.1106,
          -0.1382],
         [-0.0227, -0.1394,  0.2186, -0.5856, -0.3368,  0.2620, -0.1053,
          -0.0866]],

        [[ 0.0525, -0.2971,  0.3852, -0.5574, -0.1857,  0.0569, -0.0996,
          -0.0368],
         [ 0.0768, -0.2484,  0.3176, -0.5757, -0.2266,  0.1050, -0.0824,
           0.0197],
         [-0.0072, -0.2262,  0.2414, -0.5090, -0.2382,  0.1297, -0.0855,
          -0.0042],
         [ 0.0113, -0.2221,  0.2367, -0.5304, -0.2536,  0.1352, -0.0940,
           0.0318]]], grad_fn=<ViewBackward0>)


#positional encoding

In [28]:
class positional_encodeing(nn.Module):
  def __init__(self,d_model,max_len=5000):
    super().__init__()
    pe=torch.zeros(max_len,d_model)
    position=torch.arange(0,max_len).unsqueeze(1)
    div_term=torch.exp(
        torch.arange(0,d_model,2)*(-math.log(10000.0)/d_model)
    )

    pe[:,0::2]=torch.sin(position*div_term)

    pe[:,1::2]=torch.cos(position*div_term)

    pe=pe.unsqueeze(0)
    self.register_buffer('pe',pe)

  def forward(self,x):
    seq_len=x.size(1)

    return x+self.pe[:,:seq_len]


In [29]:
x = torch.tensor([
    [
        [0.1, 0.2, 0.3, 0.4],
        [0.5, 0.6, 0.7, 0.8],
        [0.9, 1.0, 1.1, 1.2]
    ]
])
pe = positional_encodeing(d_model=4, max_len=10)

output = pe(x)

print("Input:\n", x)
print("\nPositional encoding used:\n", pe.pe[:, :3])
print("\nOutput:\n", output)


Input:
 tensor([[[0.1000, 0.2000, 0.3000, 0.4000],
         [0.5000, 0.6000, 0.7000, 0.8000],
         [0.9000, 1.0000, 1.1000, 1.2000]]])

Positional encoding used:
 tensor([[[ 0.0000,  1.0000,  0.0000,  1.0000],
         [ 0.8415,  0.5403,  0.0100,  0.9999],
         [ 0.9093, -0.4161,  0.0200,  0.9998]]])

Output:
 tensor([[[0.1000, 1.2000, 0.3000, 1.4000],
         [1.3415, 1.1403, 0.7100, 1.8000],
         [1.8093, 0.5839, 1.1200, 2.1998]]])


#encoder

In [30]:
class feedforward(nn.Module):
  def __init__(self,d_model,d_ff):
    super().__init__()
    self.Linear1=nn.Linear(d_model,d_ff)
    self.relu=nn.ReLU()
    self.Linear2=nn.Linear(d_ff,d_model)

  def forward(self,x):
    x=self.Linear1(x)
    x=self.relu(x)
    x=self.Linear2(x)
    return x


In [31]:
class Encoder(nn.Module):
  def __init__(self,d_model,nums_heads,d_ff):
    super().__init__()
    self.mha=multi_head_attention(d_model,nums_heads)
    self.norm1=nn.LayerNorm(d_model)
    self.norm2=nn.LayerNorm(d_model)

    self.nn=feedforward(d_model,d_ff)

  def forward(self,x):
    attn_ouput=self.mha(x)
    x=self.norm1(x+attn_ouput)
    ff_output=self.nn(x)
    x=self.norm2(x+ff_output)
    return x


In [32]:
x=torch.rand(2,4,8)
encoder=Encoder(d_model=8,nums_heads=2,d_ff=32)
out=encoder(x)
print(out)

tensor([[[-0.0954,  1.3661,  1.6177, -0.4105,  0.0741, -1.7312, -0.4176,
          -0.4032],
         [-1.0385, -0.5015,  2.1621,  0.1368, -0.6255,  0.9518, -0.7552,
          -0.3300],
         [-0.8178, -0.1077,  1.0609,  0.1362,  1.0474, -1.4752, -1.1240,
           1.2802],
         [ 0.0519, -0.2406,  1.1588,  1.4810,  0.4159, -1.1195, -1.7253,
          -0.0223]],

        [[-0.3040,  1.1861,  0.2943,  1.1993,  0.0860, -1.4673, -1.5745,
           0.5800],
         [-0.4229, -0.4733,  1.4095,  1.8903, -0.4891, -0.7899, -1.0826,
          -0.0420],
         [-1.3307,  1.2058,  0.7024,  0.3915, -0.5629, -1.2072, -0.6078,
           1.4088],
         [-2.0490,  0.1514, -0.0868, -0.1385,  1.6124,  0.4222, -0.6522,
           0.7405]]], grad_fn=<NativeLayerNormBackward0>)


#TransformerEncoder using multiple encoder

In [33]:
class TransformerEncoder(nn.Module):
  def __init__(self,d_model,nums_head,d_ff,num_layers):
    super().__init__()
    self.layers=num_layers=nn.ModuleList(
        [Encoder(d_model,nums_head,d_ff) for _ in range(num_layers)]
    )

  def forward(self,x):
    for layer in self.layers:
      x=layer(x)
    return x


In [34]:
x=torch.rand(2,4,8)

encoder=TransformerEncoder(
    d_model=8,
    nums_head=2,
    d_ff=32,
    num_layers=4)

out=encoder(x)
print(out.shape)

torch.Size([2, 4, 8])


#Add Embedding + Positional Encoding

In [35]:

class Transformer(nn.Module):
  def __init__(self,vocab_size,d_model,nums_head,d_ff,num_layers):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,d_model)
    self.pos_encoding=positional_encodeing(d_model)
    self.encoder=TransformerEncoder(
        d_model,nums_head,d_ff,num_layers
    )

  def forward(self,x):
    x=self.embedding(x)
    x=self.pos_encoding(x)
    x=self.encoder(x)
    return x

#GPT block

In [36]:
class GPTBlock(nn.Module):
  def __init__(self,d_model,nums_heads,d_ff):
    super().__init__()
    self.mha=multi_head_attention(d_model,nums_heads)
    self.norm1=nn.LayerNorm(d_model)
    self.norm2=nn.LayerNorm(d_model)

    self.nn=feedforward(d_model,d_ff)

  def forward(self,x):
    attn_ouput=self.mha(x)
    x=self.norm1(x+attn_ouput)
    ff_output=self.nn(x)
    x=self.norm2(x+ff_output)
    return x


#Full GPT Model

In [37]:
class GPT(nn.Module):
  def __init__(self,
      vocab_size,
      d_model,
      num_heads,
      d_ff,
      num_layers,
      max_len=5000
    ):
      super().__init__()
      self.embedding=nn.Embedding(vocab_size,d_model)

      self.pos_encoding=positional_encodeing(d_model)

      self.layers=nn.ModuleList(
          [GPTBlock(d_model,num_heads,d_ff) for _ in range(num_layers)]

      )
      self.norm=nn.LayerNorm(d_model)
      self.fc_out=nn.Linear(d_model,vocab_size)

  def forward(self,x):

    x=self.embedding(x)
    x=self.pos_encoding(x)
    for layer in self.layers:
      x=layer(x)

    x=self.norm(x)
    logits=self.fc_out(x)
    return logits


#Create vocabulary (word → number)

#train with real dataset

In [38]:
import requests

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text


In [39]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

char_to_id = {ch:i for i,ch in enumerate(chars)}
id_to_char = {i:ch for ch,i in char_to_id.items()}

print("vocab size:", vocab_size)


vocab size: 65


In [40]:
data = torch.tensor([char_to_id[ch] for ch in text])


In [53]:
seq_len = 64

inputs = []
targets = []

for i in range(len(data) - seq_len):

    inputs.append(data[i:i+seq_len])
    targets.append(data[i+1:i+seq_len+1])

inputs = torch.stack(inputs)
targets = torch.stack(targets)

print(inputs.shape)


torch.Size([1115330, 64])


In [54]:
from torch.utils.data import Dataset

class TextDataset(Dataset):

    def __init__(self, data, seq_len):

        self.data = data
        self.seq_len = seq_len

    def __len__(self):

        return len(self.data) - self.seq_len - 1

    def __getitem__(self, idx):

        x = self.data[idx : idx + self.seq_len]

        y = self.data[idx + 1 : idx + self.seq_len + 1]

        return x, y


In [65]:
from torch.utils.data import DataLoader

seq_len = 64
batch_size = 128

dataset = TextDataset(data, seq_len)

loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=True,
    num_workers=0
)

for x_batch, y_batch in loader:
    print(x_batch.shape, y_batch.shape)
    break


torch.Size([128, 64]) torch.Size([128, 64])


In [66]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GPT(
    vocab_size=vocab_size,
    d_model=256,
    num_heads=8,
    d_ff=1024,
    num_layers=6
)

model = model.to(device)

model = torch.compile(model)

print(type(model))  # verify


<class 'torch._dynamo.eval_frame.OptimizedModule'>


In [67]:
import torch
import time
import math

# ========================
# SPEED OPTIMIZATIONS
# ========================

# Enable TF32 (free speedup on T4 / A100)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



# Optimizer and loss
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
loss_fn = torch.nn.CrossEntropyLoss()

# Mixed precision scaler
scaler = torch.cuda.amp.GradScaler()

epochs = 10
total_batches = len(loader)

print("Total batches:", total_batches)

# ========================
# TRAINING LOOP
# ========================

for epoch in range(epochs):

    model.train()

    start_time = time.time()
    total_loss = 0

    for batch_idx, (x_batch, y_batch) in enumerate(loader):

        # Move to GPU
        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        # Mixed precision forward pass
        with torch.cuda.amp.autocast():

            output = model(x_batch)

            B, T, C = output.shape

            loss = loss_fn(
                output.view(B*T, C),
                y_batch.view(B*T)
            )

        # Backward pass (scaled)
        scaler.scale(loss).backward()

        # Update weights
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

        # ========================
        # ETA calculation
        # ========================

        batches_done = batch_idx + 1
        batches_left = total_batches - batches_done

        elapsed = time.time() - start_time
        time_per_batch = elapsed / batches_done

        eta_seconds = time_per_batch * batches_left
        eta_minutes = eta_seconds / 60

        if batch_idx % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{epochs}] "
                f"Batch [{batch_idx}/{total_batches}] "
                f"Loss: {loss.item():.4f} "
                f"ETA: {eta_minutes:.2f} min"
            )

    # ========================
    # Epoch stats
    # ========================

    avg_loss = total_loss / total_batches

    perplexity = math.exp(avg_loss)

    epoch_time = (time.time() - start_time) / 60

    print(
        f"\nEpoch {epoch+1} Complete | "
        f"Avg Loss: {avg_loss:.4f} | "
        f"Perplexity: {perplexity:.2f} | "
        f"Time: {epoch_time:.2f} min\n"
    )


Using device: cuda
Total batches: 8714
Epoch [1/10] Batch [0/8714] Loss: 4.3078 ETA: 13.01 min


/tmp/ipython-input-431717910.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
/tmp/ipython-input-431717910.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [1/10] Batch [100/8714] Loss: 2.5037 ETA: 1.81 min
Epoch [1/10] Batch [200/8714] Loss: 2.3151 ETA: 1.69 min
Epoch [1/10] Batch [300/8714] Loss: 2.1743 ETA: 1.63 min
Epoch [1/10] Batch [400/8714] Loss: 2.0334 ETA: 1.60 min
Epoch [1/10] Batch [500/8714] Loss: 1.9940 ETA: 1.57 min
Epoch [1/10] Batch [600/8714] Loss: 1.9233 ETA: 1.55 min
Epoch [1/10] Batch [700/8714] Loss: 1.8437 ETA: 1.56 min
Epoch [1/10] Batch [800/8714] Loss: 1.8069 ETA: 1.58 min
Epoch [1/10] Batch [900/8714] Loss: 1.7649 ETA: 1.57 min
Epoch [1/10] Batch [1000/8714] Loss: 1.7493 ETA: 1.55 min
Epoch [1/10] Batch [1100/8714] Loss: 1.6853 ETA: 1.52 min
Epoch [1/10] Batch [1200/8714] Loss: 1.6903 ETA: 1.50 min
Epoch [1/10] Batch [1300/8714] Loss: 1.6519 ETA: 1.48 min
Epoch [1/10] Batch [1400/8714] Loss: 1.6261 ETA: 1.46 min
Epoch [1/10] Batch [1500/8714] Loss: 1.6473 ETA: 1.44 min
Epoch [1/10] Batch [1600/8714] Loss: 1.5980 ETA: 1.41 min
Epoch [1/10] Batch [1700/8714] Loss: 1.5789 ETA: 1.40 min
Epoch [1/10] Batch [180

/tmp/ipython-input-431717910.py:51: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


Epoch [2/10] Batch [100/8714] Loss: 1.2947 ETA: 1.66 min
Epoch [2/10] Batch [200/8714] Loss: 1.2789 ETA: 1.62 min
Epoch [2/10] Batch [300/8714] Loss: 1.2798 ETA: 1.64 min
Epoch [2/10] Batch [400/8714] Loss: 1.2845 ETA: 1.71 min
Epoch [2/10] Batch [500/8714] Loss: 1.3099 ETA: 1.66 min
Epoch [2/10] Batch [600/8714] Loss: 1.2746 ETA: 1.62 min
Epoch [2/10] Batch [700/8714] Loss: 1.2539 ETA: 1.59 min
Epoch [2/10] Batch [800/8714] Loss: 1.3123 ETA: 1.55 min
Epoch [2/10] Batch [900/8714] Loss: 1.2781 ETA: 1.52 min
Epoch [2/10] Batch [1000/8714] Loss: 1.2774 ETA: 1.50 min
Epoch [2/10] Batch [1100/8714] Loss: 1.2920 ETA: 1.47 min
Epoch [2/10] Batch [1200/8714] Loss: 1.2939 ETA: 1.45 min
Epoch [2/10] Batch [1300/8714] Loss: 1.2785 ETA: 1.42 min
Epoch [2/10] Batch [1400/8714] Loss: 1.2828 ETA: 1.42 min
Epoch [2/10] Batch [1500/8714] Loss: 1.2788 ETA: 1.42 min
Epoch [2/10] Batch [1600/8714] Loss: 1.2598 ETA: 1.40 min
Epoch [2/10] Batch [1700/8714] Loss: 1.2784 ETA: 1.37 min
Epoch [2/10] Batch [180

In [74]:
def generate(model, start_text, max_chars=200, temperature=0.8, top_k=40):

    model.eval()

    input_ids = torch.tensor(
        [char_to_id[ch] for ch in start_text],
        dtype=torch.long
    ).unsqueeze(0).to(device)

    for _ in range(max_chars):

        input_crop = input_ids[:, -64:]

        with torch.no_grad():
            output = model(input_crop)

        logits = output[:, -1, :] / temperature

        # Top-k filtering
        values, indices = torch.topk(logits, top_k)
        probs = torch.softmax(values, dim=-1)

        next_index = torch.multinomial(probs, num_samples=1)
        next_id = torch.gather(indices, -1, next_index)

        input_ids = torch.cat([input_ids, next_id], dim=1)


    return "".join([id_to_char[i] for i in input_ids[0].cpu().tolist()])


In [83]:
print(generate(model, "Nay, gentle "))

Nay, gentle queen,
And yet I should be accompant,
And aid her with that consume
As with a lip.

MERCUTIO:
That we came fix'd, keep with well
Be broken forth to some place, coal-borough we died our officer.

MISTR


In [84]:
torch.save(model.state_dict(), "gpt_model.pth")


In [85]:
import pickle

with open("vocab.pkl", "wb") as f:
    pickle.dump({
        "char_to_id": char_to_id,
        "id_to_char": id_to_char
    }, f)


In [86]:
from google.colab import files

files.download("gpt_model.pth")
files.download("vocab.pkl")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

NameError: name 'sher' is not defined